In [1]:
# Import libraries
import sqlite3
import os
import logging
import pandas as pd
from dataframeinspector import DataFrameInspector

In [2]:
# Create connection to the db
conn = sqlite3.connect("dev\cademycode.db")

# Create a cursor object
cur = conn.cursor()

In [3]:
# Check the tables in the db
cur.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

[('cademycode_students',),
 ('cademycode_courses',),
 ('cademycode_student_jobs',)]

# Clean the data

In [4]:
students_df = pd.read_sql_query("SELECT * FROM cademycode_students;", conn)
courses_df = pd.read_sql_query("SELECT * FROM cademycode_courses;", conn)
student_jobs_df = pd.read_sql_query("SELECT * FROM cademycode_student_jobs;", conn)

In [20]:
courses_df

,career_path_id,career_path_name,hours_to_complete
0,1,data scientist,20
1,2,data engineer,20
2,3,data analyst,12
3,4,software engineering,25
4,5,backend engineer,18
5,6,frontend engineer,20
6,7,iOS developer,27
7,8,android developer,27
8,9,machine learning engineer,35
9,10,ux/ui designer,15


In [5]:
students_df_obj = DataFrameInspector(students_df)
students_df_obj.convert_column_type('dob', 'date_only')
int_columns = ['job_id', 'num_course_taken', 'current_career_path_id']
students_df_obj.convert_column_type(int_columns, 'int')
students_df_obj.convert_column_type('time_spent_hrs', 'float')
students_df_obj.extract_json_column('contact_info')
students_df_obj.examine()

✅ Column 'dob' converted to datetime64[ns] with no time.
⚠️ Column 'job_id' contains NaN after numeric conversion. Filling with 0.
✅ Column 'job_id' safely converted to int.
⚠️ Column 'num_course_taken' contains NaN after numeric conversion. Filling with 0.
✅ Column 'num_course_taken' safely converted to int.
⚠️ Column 'current_career_path_id' contains NaN after numeric conversion. Filling with 0.
✅ Column 'current_career_path_id' safely converted to int.
✅ Column 'time_spent_hrs' converted to float.
✅ Extracted keys ['mailing_address', 'email'] from 'contact_info' and dropped the original column.


,Columns,Data Types,Null Values,Unique Values
0,uuid,int64,0,5000
1,name,object,0,4998
2,dob,datetime64[ns],0,4492
3,sex,object,0,3
4,job_id,int32,0,9
5,num_course_taken,int32,0,16
6,current_career_path_id,int32,0,11
7,time_spent_hrs,float64,471,2192
8,mailing_address,object,0,5000
9,email,object,0,5000


In [6]:
courses_df_obj = DataFrameInspector(courses_df)
courses_df_obj.examine()

,Columns,Data Types,Null Values,Unique Values
0,career_path_id,int64,0,10
1,career_path_name,object,0,10
2,hours_to_complete,int64,0,7


In [7]:
student_jobs_df_obj = DataFrameInspector(student_jobs_df)
student_jobs_df_obj.convert_column_type('avg_salary', 'float')
student_jobs_df_obj.drop_duplicates(subset=['job_id'])
student_jobs_df_obj.examine()

✅ Column 'avg_salary' converted to float.
✅ Dropped 3 duplicate row(s). 10 row(s) remain.


,Columns,Data Types,Null Values,Unique Values
0,job_id,int64,0,10
1,job_category,object,0,10
2,avg_salary,float64,0,9


In [8]:
# Convert back to dfs
students_df_clean = students_df_obj.df
courses_df_clean = courses_df_obj.df
student_jobs_df_clean = student_jobs_df_obj.df

In [9]:
# Merge the dataframes into a single dataframe
# 1. Merge students_df_clean with courses_df_clean
merged_student_courses_df = pd.merge(
    students_df_clean,
    courses_df_clean,
    left_on='current_career_path_id',
    right_on='career_path_id',
    how='left',
    suffixes=('_students', '_courses')
)

merged_student_courses_df.drop(columns=['career_path_id'], inplace=True)

# 2. Merge merged_student_courses_df with student_jobs_df_clean
final_df = pd.merge(
    merged_student_courses_df,
    student_jobs_df_clean,
    on='job_id',
    how='left'
)

In [10]:
desired_columns_order = ['uuid', 'job_id', 'current_career_path_id', 'dob', 'name', 'sex', 'email', 'mailing_address', 'num_course_taken', 'time_spent_hrs', 'career_path_name', 'hours_to_complete', 'job_category', 'avg_salary']

In [11]:
final_df = final_df[desired_columns_order]
final_df

,uuid,job_id,current_career_path_id,dob,name,sex,email,mailing_address,num_course_taken,time_spent_hrs,career_path_name,hours_to_complete,job_category,avg_salary
0,1,7,1,1943-07-03,Annabelle Avery,F,annabelle_avery9376@woohoo.com,"303 N Timber Key, Irondale, Wisconsin, 84736",6,4.99,data scientist,20.0,HR,80000.0
1,2,7,8,1991-02-07,Micah Rubio,M,rubio6772@hmail.com,"767 Crescent Fair, Shoals, Indiana, 37439",5,4.40,android developer,27.0,HR,80000.0
2,3,7,8,1989-12-07,Hosea Dale,M,hosea_dale8084@coldmail.com,"P.O. Box 41269, St. Bonaventure, Virginia, 83637",8,6.74,android developer,27.0,HR,80000.0
3,4,6,9,1988-07-31,Mariann Kirk,F,kirk4005@hmail.com,"517 SE Wintergreen Isle, Lane, Arkansas, 82242",7,12.31,machine learning engineer,35.0,education,61000.0
4,5,7,3,1963-08-31,Lucio Alexander,M,alexander9810@hmail.com,"18 Cinder Cliff, Doyles borough, Rhode Island,...",14,5.64,data analyst,12.0,HR,80000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,4996,5,2,1967-07-07,Quentin van Harn,N,vanharn2778@woohoo.com,"591 Blue Berry, Coulee, Illinois, 65199",5,13.82,data engineer,20.0,financial services,135000.0
4996,4997,4,1,1964-11-03,Alejandro van der Sluijs,M,alejandro4080@coldmail.com,"30 Iron Divide, Pewaukee village, California, ...",13,7.86,data scientist,20.0,creative,66000.0
4997,4998,8,3,2004-11-25,Brock Mckenzie,M,brock_mckenzie2025@inlook.com,"684 Rustic Rest Avenue, Carmine, California, 5...",10,12.10,data analyst,12.0,student,10000.0
4998,4999,3,5,1943-02-12,Donnetta Dillard,N,dillard7526@inlook.com,"900 Indian Oval, Euclid, Iowa, 59683",6,14.86,backend engineer,18.0,software developer,110000.0


# Create the clean database

## Create the master table

In [12]:
# Connect to the "clean" database (this will create the file if it doesn't exist)
conn_clean = sqlite3.connect(r"dev\cademycode_clean.db")

# Write final_df to a new table
final_df.to_sql(
    "cademycode_master_students_table",
    conn_clean,
    if_exists="replace",
    index=False
)

conn_clean.commit()

print("✅ Clean database created and table written successfully.")

✅ Clean database created and table written successfully.


## Create the rest of the tables

### Students

In [13]:
# Write final_df to a new table
students_df_clean.to_sql(
    "students",
    conn_clean,
    if_exists="replace",
    index=False
)

conn_clean.commit()
print("✅ Students table created and populated successfully.")

✅ Students table created and populated successfully.


In [14]:
# Check results
# curs_clean = conn_clean.cursor()
# students = pd.read_sql_query("SELECT * FROM students;", conn_clean)
# students

### Courses

In [15]:
curs_clean = conn_clean.cursor()
curs_clean.execute("""
CREATE TABLE IF NOT EXISTS courses (
    career_path_id TEXT PRIMARY KEY, 
    career_path_name TEXT,
    hours_to_complete REAL
);
""")

# Prepare the insert query with placeholders
insert_query = """
INSERT OR REPLACE INTO courses (
    career_path_id, career_path_name, hours_to_complete
) VALUES (?, ?, ?);
"""

# Execute for all rows
curs_clean.executemany(
    insert_query,
    courses_df_clean.values.tolist()  # converts DataFrame to list of tuples
)

conn_clean.commit()
print("✅ Courses table created and data inserted successfully.")

✅ Courses table created and data inserted successfully.


In [16]:
# Check the results
# curs_clean.execute("SELECT * FROM courses").fetchall()

### Students jobs

In [17]:
curs_clean = conn_clean.cursor()
curs_clean.execute("""
CREATE TABLE IF NOT EXISTS students_jobs (
    job_id TEXT PRIMARY KEY, 
    job_category TEXT,
    avg_salary REAL
);
""")

# Prepare the insert query with placeholders
insert_query = """
INSERT OR REPLACE INTO students_jobs (
    job_id, job_category, avg_salary
) VALUES (?, ?, ?);
"""

# Execute for all rows
curs_clean.executemany(
    insert_query,
    student_jobs_df_clean.values.tolist()  # converts DataFrame to list of tuples
)

conn_clean.commit()
print("✅ Students jobs table created and data inserted successfully.")

✅ Students jobs table created and data inserted successfully.


In [18]:
# Check the results
# curs_clean.execute("SELECT * FROM students_jobs").fetchall()

# Export the master table to a csv file

In [19]:
with open(r"dev\cademycode_master_students_table.csv", "w", encoding="utf-8") as f:
    final_df.to_csv(f, index=False)

PermissionError: [Errno 13] Permission denied: 'dev\\cademycode_master_students_table.csv'